# Interactive Map: Launch Site Location Analysis (Folium)

This notebook builds an interactive Folium map to visualize SpaceX Falcon 9 launch sites and launch outcomes (success vs. failure). It also computes simple proximity distances (coastline / railway / highway / nearest city) for a representative launch site to reason about why launch sites tend to be located near key infrastructure and open water.

**Inputs (from earlier steps):** `../data/processed/03_dataset_part_2.csv`

**Outputs (this step):** `../data/processed/06_launch_site_map.html` (interactive map export)

---

## 1. Setup

We keep the original lab computation, but remove lab-specific mechanics (e.g., downloading datasets in-notebook).

In [1]:
from pathlib import Path

import pandas as pd
import folium

# Paths
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

INPUT_CSV = PROCESSED_DIR / '03_dataset_part_2.csv'
OUTPUT_MAP_HTML = PROCESSED_DIR / '06_launch_site_map.html'

# Folium plugins / features used later
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

## 2. Load the processed dataset

Step 03 produced a cleaned dataset that already includes launch site names and coordinates. To keep the downstream code from the original intact, we rename a few columns in-memory.

In [2]:
df = pd.read_csv(INPUT_CSV)

# Keep only the columns needed for mapping (renamed to match the original lab variables)

spacex_df = (
    df.rename(
        columns = {
            'LaunchSite': 'Launch Site',
            'Latitude': 'Lat',
            'Longitude': 'Long',
            'Class': 'class',
        }
    )[['Launch Site', 'Lat', 'Long', 'class']]
    .copy()
)

spacex_df.head()

,Launch Site,Lat,Long,class
0,CCSFS SLC 40,28.561857,-80.577366,0
1,CCSFS SLC 40,28.561857,-80.577366,0
2,CCSFS SLC 40,28.561857,-80.577366,0
3,VAFB SLC 4E,34.632093,-120.610829,0
4,CCSFS SLC 40,28.561857,-80.577366,0


## 3. Derive unique launch sites

We create a small `launch_sites_df` with one row per launch site, using the first observed coordinate.

In [3]:
# We already selected relevant sub-columns and renamed the DataFrame as spacex_df
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index = False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCSFS SLC 40,28.561857,-80.577366
1,KSC LC 39A,28.608058,-80.603956
2,VAFB SLC 4E,34.632093,-120.610829


## 4. Base map (reference point)

We start from NASA Johnson Space Center for a familiar reference point, then zoom out to include all SpaceX launch sites.

In [4]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location = nasa_coordinate, zoom_start = 10)

In [5]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(
    nasa_coordinate,
    radius = 1000,
    color = '#d35400',
    fill = True).add_child(folium.Popup('NASA Johnson Space Center'))

# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon = DivIcon(
        icon_size = (20, 20),
        icon_anchor = (0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

## 5. Mark all launch sites

Each launch site is added as:

- a circle with a popup label, and

- a text label (DivIcon) for readability at different zoom levels.

In [6]:
# Initial the map
site_map = folium.Map(location = nasa_coordinate, zoom_start = 5)

# For each launch site, add a Circle + popup, and a text label marker
for _, site in launch_sites_df.iterrows():
    site_coordinate = [site['Lat'], site['Long']]
    site_name = site['Launch Site']

    # Circle + popup
    site_map.add_child(
        folium.Circle(
            location = site_coordinate,
            radius = 1000,
            color = "#d35400",
            fill = True,
            fill_color = "#d35400",
            fill_opacity = 0.2,
        ).add_child(folium.Popup(site_name))
    )

    # Text label
    site_map.add_child(
        folium.map.Marker(
            site_coordinate,
            icon = DivIcon(
                icon_size = (250, 20),
                icon_anchor = (0, 0),
                html = f'<div style="font-size: 12px; color:#d35400;"><b>{site_name}</b></div>',
            ),
        )
    )

site_map

## 6. Mark success vs. failure outcomes

We overlay individual launches using a MarkerCluster. Successful landings are **green** and failures are **red**.

In [7]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
80,CCSFS SLC 40,28.561857,-80.577366,1
81,CCSFS SLC 40,28.561857,-80.577366,1
82,CCSFS SLC 40,28.561857,-80.577366,1
83,CCSFS SLC 40,28.561857,-80.577366,1
84,CCSFS SLC 40,28.561857,-80.577366,1
85,KSC LC 39A,28.608058,-80.603956,1
86,KSC LC 39A,28.608058,-80.603956,1
87,KSC LC 39A,28.608058,-80.603956,1
88,CCSFS SLC 40,28.561857,-80.577366,1
89,CCSFS SLC 40,28.561857,-80.577366,1


In [8]:
marker_cluster = MarkerCluster()

In [9]:
site_map.add_child(marker_cluster)

In [10]:
# Function to assign color to launch outcome
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
80,CCSFS SLC 40,28.561857,-80.577366,1,green
81,CCSFS SLC 40,28.561857,-80.577366,1,green
82,CCSFS SLC 40,28.561857,-80.577366,1,green
83,CCSFS SLC 40,28.561857,-80.577366,1,green
84,CCSFS SLC 40,28.561857,-80.577366,1,green
85,KSC LC 39A,28.608058,-80.603956,1,green
86,KSC LC 39A,28.608058,-80.603956,1,green
87,KSC LC 39A,28.608058,-80.603956,1,green
88,CCSFS SLC 40,28.561857,-80.577366,1,green
89,CCSFS SLC 40,28.561857,-80.577366,1,green


In [11]:
for _, record in spacex_df.iterrows():
    coordinate = [record['Lat'], record['Long']]

    # Optional popup text (nice for debugging)
    outcome = 'Success' if record['class'] == 1 else 'Failure'
    popup_text = f'{record['Launch Site']} — {outcome}'

    marker = folium.Marker(
        location = coordinate,
        icon = folium.Icon(
            color = 'white',
            icon_color = record['marker_color'],
            icon = 'info-sign'
        ),
        popup = folium.Popup(popup_text, max_width = 300),
    )
    marker_cluster.add_child(marker)

site_map

## 7. Map helper: live mouse coordinates

The MousePosition plugin is convenient when you want to pick coordinates directly from the map (e.g., coastline points).

In [12]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = 'function(num) {return L.Util.formatNum(num, 5);};'
mouse_position = MousePosition(
    position = 'topright',
    separator = ' Long: ',
    empty_string = 'NaN',
    lng_first = False,
    num_digits = 20,
    prefix = 'Lat:',
    lat_formatter = formatter,
    lng_formatter = formatter,
)

site_map.add_child(mouse_position)
site_map

## 8. Distance utility (Haversine)

We use a simple great-circle distance approximation (Haversine) to compute distances between two coordinates.

In [13]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

def dist_km(p1, p2):
    return calculate_distance(p1[0], p1[1], p2[0], p2[1])

## 9. Proximity example for one launch site

To keep analysis reproducible, the coordinates below are **recorded** examples near CCAFS LC-40:

- coastline point (as in the original lab),

- a nearby rail facility point from NASA/KSC documentation,

- a major highway connection (SR 401 / Port Canaveral area),

- nearby city coordinates (Cape Canaveral).

Distances are computed in kilometers using `calculate_distance` function.

In [14]:
# Select a launch site to analyze (representative example)
target = 'SLC 40'
launch_site_row = launch_sites_df[
    launch_sites_df['Launch Site'].str.contains(target, case = False, na = False)
]
assert not launch_site_row.empty, (
    f'Launch site containing \'{target}\' not found. '
    f'Available: {launch_sites_df['Launch Site'].unique().tolist()}'
)

launch_site = launch_site_row['Launch Site'].iloc[0]  # actual name
launch_site_lat = float(launch_site_row['Lat'].iloc[0])
launch_site_long = float(launch_site_row['Long'].iloc[0])
launch_site_coordinate = [launch_site_lat, launch_site_long]

# Coastline coordinate (picked using MousePosition on the map)
coastline_coordinate = [28.56180, -80.56769]

# Railway coordinate (NASA/KSC rail system area, picked using MousePosition on the map)
railway_coordinate = [28.56180, -80.58725]

# Highway coordinate (SR 401 / Port Canaveral area, picked using MousePosition on the map)
highway_coordinate = [28.56180, -80.57055]

# Nearby city coordinate (Cape Canaveral, FL, picked using MousePosition on the map)
city_coordinate = [28.38800, -80.60567]

# Compute distances (km)
distance_coastline_km = dist_km(launch_site_coordinate, coastline_coordinate)
distance_railway_km = dist_km(launch_site_coordinate, railway_coordinate)
distance_highway_km = dist_km(launch_site_coordinate, highway_coordinate)
distance_city_km = dist_km(launch_site_coordinate, city_coordinate)

pd.DataFrame(
    {
        'feature': ['coastline', 'railway', 'highway', 'city'],
        'latitude': [
            coastline_coordinate[0],
            railway_coordinate[0],
            highway_coordinate[0],
            city_coordinate[0],
        ],
        'longitude': [
            coastline_coordinate[1],
            railway_coordinate[1],
            highway_coordinate[1],
            city_coordinate[1],
        ],
        'distance_km': [
            distance_coastline_km,
            distance_railway_km,
            distance_highway_km,
            distance_city_km,
        ],
    }
).sort_values('distance_km')

,feature,latitude,longitude,distance_km
2,highway,28.5618,-80.57055,0.665908
0,coastline,28.5618,-80.56769,0.945302
1,railway,28.5618,-80.58725,0.965622
3,city,28.3880,-80.60567,19.535107


In [15]:
# Coastline marker
site_map.add_child(
    folium.Marker(
        coastline_coordinate,
        icon = folium.Icon(color = 'blue', icon='info-sign'),
        popup = folium.Popup('Closest coastline point (selected)', max_width = 250),
    )
)

# Distance label
distance_marker = folium.Marker(
    coastline_coordinate,
    icon = DivIcon(
        icon_size = (150, 20),
        icon_anchor = (0, 0),
        html = '<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>'
        % '{:10.2f} KM'.format(distance_coastline_km),
    ),
)
site_map.add_child(distance_marker)

site_map

In [16]:
coordinates = [launch_site_coordinate, coastline_coordinate]
lines = folium.PolyLine(locations = coordinates, weight = 2)
site_map.add_child(lines)

site_map

In [17]:
# Add markers + distance labels for other proximity features
def add_distance_marker(feature_name: str, coordinate: list[float], distance_km: float):
    # Add marker
    folium.Marker(
        coordinate,
        icon = folium.Icon(color = 'blue', icon = 'info-sign'),
        popup = f'{feature_name.title()} (≈ {distance_km:.2f} km)',
    ).add_to(site_map)

    # Text label for distance (placed at the feature location)
    folium.map.Marker(
        coordinate,
        icon = DivIcon(
            icon_size = (250, 20),
            icon_anchor = (0, 0),
            html = f'<div style = "font-size: 12px; color: #2c3e50;"><b>{feature_name}: {distance_km:.2f} km</b></div>',
        ),
    ).add_to(site_map)

    # Line from launch site to feature
    folium.PolyLine(
        [launch_site_coordinate, coordinate],
        weight = 2,
        opacity = 0.6
    ).add_to(site_map)

add_distance_marker('railway', railway_coordinate, distance_railway_km)
add_distance_marker('highway', highway_coordinate, distance_highway_km)
add_distance_marker('city', city_coordinate, distance_city_km)

site_map

## 10. Export

Export the interactive map as HTML artifact.

In [18]:
# Save the final map to an HTML file
site_map.save(str(OUTPUT_MAP_HTML))

---